In [3]:
%pip install -qU \
    "langchain>=1.0,<2.0" \
    langchain-core \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-chroma \
    langchain-google-genai \
    sentence-transformers \
    chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/6

In [4]:
"""
Our motivating example today will be a simple museum heist
This isn't designed to be challenging but it'll allow us to use an LLM
to generate responses from our "evidence"
"""
case_files = [
    {
        "source": "security_log.md",
        "category": "security",
        "text": """
8:47 PM — Guard Elena Ruiz began her east-wing patrol.

9:12 PM — Staff badge B-17 opened the east gallery door.

9:14 PM — Motion was detected near Exhibit 14.

9:26 PM — Badge B-17 exited through the restoration corridor.

9:33 PM — The east gallery display-case alarm activated.
        """.strip(),
    },
    {
        "source": "staff_access_records.md",
        "category": "access",
        "text": """
Badge B-17 is assigned to restoration specialist Marcus Bell.

Restoration staff may enter the east gallery after closing only when an approved work order
covers the employee, time, room, and exhibit involved.

Security staff may patrol the gallery but may not open exhibit cases without an emergency
authorization.
        """.strip(),
    },
    {
        "source": "witness_statement_elena.md",
        "category": "witness",
        "text": """
Guard Elena Ruiz reported seeing a person wearing a blue maintenance jacket near the east
gallery service elevator at approximately 9:20 PM.

She could not see the person's face or badge number.

She heard the sound of a metal cart moving toward the restoration corridor a few minutes later.
        """.strip(),
    },
    {
        "source": "approved_work_orders.md",
        "category": "operations",
        "text": """
Work order WO-184 authorized conservator Priya Shah to inspect Exhibit 12 from 7:00 PM until
8:30 PM.

No approved work order covered Exhibit 14 on the night of the incident.
        """.strip(),
    },
    {
        "source": "exhibit_catalog.md",
        "category": "catalog",
        "text": """
Exhibit 14 contained the Moonstone Compass, an 1887 brass-and-glass navigation instrument.

At 9:34 PM, responding security staff found the display case open and the Moonstone Compass
missing.

The catalog does not contain information about a getaway vehicle or suspect transportation.
        """.strip(),
    },
]

In [5]:
"""
Next we'll create a LangChain Document which is a standardized representation of
our documents. It doesn't really matter what or where the original document came from
it'll be morphed into this form.

Document(
  page_content="the searchable text",
  metadata ={"source":....}
)
"""
from langchain_core.documents import Document

documents = [
    Document(
        page_content = file["text"],
        metadata = {
            "source": file["source"],
            "category": file["category"]
        }
    )
    for file in case_files
]

first_document = documents[0]

print("Page Content")
print(first_document.page_content)

print("Metadata")
print(first_document.metadata)

Page Content
8:47 PM — Guard Elena Ruiz began her east-wing patrol.

9:12 PM — Staff badge B-17 opened the east gallery door.

9:14 PM — Motion was detected near Exhibit 14.

9:26 PM — Badge B-17 exited through the restoration corridor.

9:33 PM — The east gallery display-case alarm activated.
Metadata
{'source': 'security_log.md', 'category': 'security'}


In [7]:
"""
Next step is cleaning and chunking. As a reminder, we can clean the text by removing
white spaces, formatting errors, etc. We then Chunk the data into more manageable
bite-size pieces. Why? We lose context in our models and our vector embeddings
if the size of the document is too large (token limits)
"""

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=220,
    chunk_overlap=40
)

# To create the chunks we'll apply the splitter to the documents
chunks = splitter.split_documents(documents)
# This splits the LANGCHAIN DOCUMENTS

for index,chunk in enumerate(chunks):
  print("Chunk: ", index)
  print("Metadata: ", chunk.metadata)
  print("Text:", chunk.page_content)

Chunk:  0
Metadata:  {'source': 'security_log.md', 'category': 'security'}
Text: 8:47 PM — Guard Elena Ruiz began her east-wing patrol.

9:12 PM — Staff badge B-17 opened the east gallery door.

9:14 PM — Motion was detected near Exhibit 14.
Chunk:  1
Metadata:  {'source': 'security_log.md', 'category': 'security'}
Text: 9:26 PM — Badge B-17 exited through the restoration corridor.

9:33 PM — The east gallery display-case alarm activated.
Chunk:  2
Metadata:  {'source': 'staff_access_records.md', 'category': 'access'}
Text: Badge B-17 is assigned to restoration specialist Marcus Bell.

Restoration staff may enter the east gallery after closing only when an approved work order
covers the employee, time, room, and exhibit involved.
Chunk:  3
Metadata:  {'source': 'staff_access_records.md', 'category': 'access'}
Text: Security staff may patrol the gallery but may not open exhibit cases without an emergency
authorization.
Chunk:  4
Metadata:  {'source': 'witness_statement_elena.md', 'categ

In [8]:
"""
Our next step is to create our embeddings
We'll use a langchain version of our same embedding model to create an embedding
"""
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# When we use this it'll give us our 384-dimensional vectors

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
example_sentences = [
    "Badge B-17 entered the gallery",
    "The Cafe serves breakfast till 10"
]

# Embed Sentences (Documents)
example_vectors = embeddings.embed_documents(example_sentences)

print(len(example_vectors))
print(example_vectors[0][:10])

2
[0.0014349855482578278, 0.07133173942565918, 0.002016632352024317, -0.019041629508137703, 0.05683809146285057, 0.08653838187456131, 0.07330192625522614, -0.06257476657629013, -0.05900098755955696, -0.04111702740192413]


In [11]:
"""
Embed and stor all of our chunks in a ChromaDB

Recall, LangChain is used to abstract the process of using an LM in your application.
In our case that means we can swap out different implementations of VectorDBs (chroma
or pinecone for example) and have minimal code changes
"""

from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    # Pass our chunks to the DB to store
    documents = chunks,
    # Provide our embedding model
    embedding=embeddings,
    # Provide a collection name
    collection_name="museum_case_files"
)

vector_store

In [14]:
"""
At this point we are not yet doing RAG but let's do a similarity search
which will be the first part of the Query half of RAG.

Two halves of RAG (for now):
Index:
  Get Docs -> Clean -> Chunk -> Embed -> Store
Query:
  Similarity Search -> Retrieve Docs -> Pass to Prompt -> Pass to LLM -> Generate and Return Response

"""

# Let's practice a similarity search to make sure our vector db is good to go
question = "Which badge entered the east gallery after 9 PM?"

matches = vector_store.similarity_search(
    question,
    k=3
)

matches

[Document(id='68261f8a-be9d-479d-89cb-902b3af690ad', metadata={'source': 'security_log.md', 'category': 'security'}, page_content='9:26 PM — Badge B-17 exited through the restoration corridor.\n\n9:33 PM — The east gallery display-case alarm activated.'),
 Document(id='6fc92063-e1e8-4024-90c4-4e0ccbe6c2c9', metadata={'source': 'security_log.md', 'category': 'security'}, page_content='9:26 PM — Badge B-17 exited through the restoration corridor.\n\n9:33 PM — The east gallery display-case alarm activated.'),
 Document(id='32dfa108-eec7-4e46-bfcd-a408df2e2eab', metadata={'source': 'staff_access_records.md', 'category': 'access'}, page_content='Badge B-17 is assigned to restoration specialist Marcus Bell.\n\nRestoration staff may enter the east gallery after closing only when an approved work order\ncovers the employee, time, room, and exhibit involved.')]

In [15]:
"""
We're going to narrow the scope of the vector store
Vector DBs can be used for a lot of different things beyond what we're talking
about here, so we're going to convert this into something called a Retriever

A Retriever retrieves documents
"""

retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

retrieved_documents = retriever.invoke(
    "Who entered the east gallery after 9PM"
)

retrieved_documents

[Document(id='54a3aa44-fe48-414a-a327-e4eb70724276', metadata={'category': 'security', 'source': 'security_log.md'}, page_content='8:47 PM — Guard Elena Ruiz began her east-wing patrol.\n\n9:12 PM — Staff badge B-17 opened the east gallery door.\n\n9:14 PM — Motion was detected near Exhibit 14.'),
 Document(id='4962d5b3-1f69-41c5-b90a-214f4e27e702', metadata={'source': 'security_log.md', 'category': 'security'}, page_content='8:47 PM — Guard Elena Ruiz began her east-wing patrol.\n\n9:12 PM — Staff badge B-17 opened the east gallery door.\n\n9:14 PM — Motion was detected near Exhibit 14.'),
 Document(id='68261f8a-be9d-479d-89cb-902b3af690ad', metadata={'source': 'security_log.md', 'category': 'security'}, page_content='9:26 PM — Badge B-17 exited through the restoration corridor.\n\n9:33 PM — The east gallery display-case alarm activated.'),
 Document(id='6fc92063-e1e8-4024-90c4-4e0ccbe6c2c9', metadata={'source': 'security_log.md', 'category': 'security'}, page_content='9:26 PM — Badge

In [17]:
"""
Here we've received a list of LangChain documents and I want
to convert it to a more human-readble format so I can pass it as
context to my LLM
"""

context = "\n\n".join(
    f"Source: {document.metadata["source"]}\n"
    f"Category: {document.metadata["category"]}\n"
    f"Evidence: {document.page_content}"
    for document in retrieved_documents
)

print(context)

Source: security_log.md
Category: security
Evidence: 8:47 PM — Guard Elena Ruiz began her east-wing patrol.

9:12 PM — Staff badge B-17 opened the east gallery door.

9:14 PM — Motion was detected near Exhibit 14.

Source: security_log.md
Category: security
Evidence: 8:47 PM — Guard Elena Ruiz began her east-wing patrol.

9:12 PM — Staff badge B-17 opened the east gallery door.

9:14 PM — Motion was detected near Exhibit 14.

Source: security_log.md
Category: security
Evidence: 9:26 PM — Badge B-17 exited through the restoration corridor.

9:33 PM — The east gallery display-case alarm activated.

Source: security_log.md
Category: security
Evidence: 9:26 PM — Badge B-17 exited through the restoration corridor.

9:33 PM — The east gallery display-case alarm activated.


In [18]:
"""
We preformed that operation on a single set of retrieved docs but let's
make it a reusable function
"""

def format_documents(documents):
  return "\n\n".join(
    f"Source: {document.metadata["source"]}\n"
    f"Category: {document.metadata["category"]}\n"
    f"Evidence: {document.page_content}"
    for document in retrieved_documents
  )

In [20]:
"""
At this point we have the setup to insert documents and retrieve them
now we need to connect to an LLM. We're using Gemini 3.1 flash lite for this
BUT be warned, if you're paying then you are the product (Google uses your
queries in it's own training data). Dont use this method for secure things.
They'll only give you security if you pay for it
"""
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

# Let's get our secret
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Load the model

llm = ChatGoogleGenerativeAI(
    model = "gemini-3.1-flash-lite",
    # Temperature controls how determinstic vs creative the model can be
    temperature = 0.0,
    max_tokens = 500,
    max_retries = 2
)

In [23]:
"""
We can verify that we're actually connected to an LLM and take a look at what it responds
"""
text_response = llm.invoke(
    # "Reply with exactly: Gemini Connection Successful"
    "Tell me a joke about penguins"
)

print(text_response.content)

[{'type': 'text', 'text': 'Why don’t you see penguins in Great Britain?\n\nBecause they’re afraid of **Wales**!', 'extras': {'signature': 'EjQKMgERTTIPvgSVpzqGWTws0Zz8NgCkJ066txG4b9xORYnoMWEjT5xyvhQhR1dIKi33P4si'}}]


In [25]:
"""
To Pass the information we have as context as well as the question, you need a
prompt template. This will allow us to create a reusable prompt with the
same set of instructions for every request

Inlcude things like person, guardrails, etc
"""
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an investigator reviewing museuem case files.

Answer the question using only the evidence supplied below.

If the evidence does not contain the answer, respond exactly:
"The available case files do not contain that information."

Identify the source or sources that support the answer

Distinguish between:
- what the files directly establish,
- what they merely suggest,
- and what remains unknown.

Do not invent evidence.
Do not state that a person committed the theft unless the supplied evidence proves it

EVIDENCE:
{context}

QUESTION:
{question}
""")

In [27]:
retrieved_documents = retriever.invoke(question)
context = format_documents(retrieved_documents)

# Pass the values to the prompt
formatted_prompt = prompt.invoke({
    "context": context,
    "question": question
})

print(formatted_prompt.to_string())

Human: 
You are an investigator reviewing museuem case files.

Answer the question using only the evidence supplied below.

If the evidence does not contain the answer, respond exactly:
"The available case files do not contain that information."

Identify the source or sources that support the answer

Distinguish between:
- what the files directly establish,
- what they merely suggest,
- and what remains unknown.

Do not invent evidence.
Do not state that a person committed the theft unless the supplied evidence proves it

EVIDENCE:
Source: security_log.md
Category: security
Evidence: 9:26 PM — Badge B-17 exited through the restoration corridor.

9:33 PM — The east gallery display-case alarm activated.

Source: security_log.md
Category: security
Evidence: 9:26 PM — Badge B-17 exited through the restoration corridor.

9:33 PM — The east gallery display-case alarm activated.

Source: staff_access_records.md
Category: access
Evidence: Badge B-17 is assigned to restoration specialist Marcu

In [29]:
"""
At this point we have performed the Index half (loading and embedding documents)
Now we need to loop the entire query half together, we'll do this manually
at first
"""

question = "Who entered the east gallery?"

# Step 1: RETRIEVE the docs
retrieved_documents = retriever.invoke(question)

# Prepare the context
context = format_documents(retrieved_documents)

# Fill out the prompt
formatted_prompt = prompt.invoke({
    "context": context,
    "question": question
})

# Step 2: GENERATE an answer from the evidence
response = llm.invoke(formatted_prompt)

print(response.content)

[{'type': 'text', 'text': 'Based on the provided evidence:\n\n**What the files directly establish:**\nThe individual who entered the east gallery at 9:12 PM used staff badge B-17. According to the staff access records, badge B-17 is assigned to restoration specialist Marcus Bell.\n\n**What the files suggest:**\nThe files suggest that Marcus Bell entered the east gallery, as his assigned badge was used to access the room.\n\n**What remains unknown:**\nThe files do not explicitly confirm that Marcus Bell was the person physically holding or using the badge at 9:12 PM.\n\n**Sources:**\n*   `security_log.md`\n*   `staff_access_records.md`', 'extras': {'signature': 'EjQKMgERTTIP8SfhBUwxmOjGga/COiDIuFzTwzEvn73ZzJLw4WFgthGKNbFWRBiaVliGaWNR'}}]


In [31]:
"""
We're almost done, technically at this point we have completed a RAG flow.
But it's kinda tedious to pass one output to the next and what happens if variables
get reassigned in the middle or there's a disconnect or anything like that?

We're going to rebuild this using a RunnableSequence which is a series of steps
defined with LCEL (LangChain Expression Language) that represents the exact
steps we just took
"""

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# We'll define our actual rag_chain
# Looking ahead this will receive the question as an input via invoke

rag_chain = (
    # Receives the Question as an input
    {
        # This takes the previous input (the question) and passes it to
        # the retriever (which fetches docs) and then passes that through the
        # "pipe" operator | to the format_documents function
        "context": retriever | format_documents,
        # Runnable Passthrough means take the previous input (the question) and
        # return it
        "question": RunnablePassthrough() # Should be just the question itself
    }
    # The context and question get passed to the prompt for creation
    | prompt
    # The correct prompt gets passed to the LLM for response generation
    | llm
    # This output parser just makes the output string look cleaner
    | StrOutputParser()
)

In [36]:
answer = rag_chain.invoke(
    "Who entered the east gallery?"
)

print(answer)

Based on the provided evidence:

**What the files directly establish:**
The individual who entered the east gallery at 9:12 PM used staff badge B-17. According to the staff access records, badge B-17 is assigned to restoration specialist Marcus Bell.

**What the files suggest:**
The files suggest that Marcus Bell entered the east gallery, as his assigned badge was used to access the room.

**What remains unknown:**
The files do not explicitly confirm that Marcus Bell was the person physically holding or using the badge at 9:12 PM.

**Sources:**
*   `security_log.md`
*   `staff_access_records.md`


In [37]:
missing_answer = rag_chain.invoke(
    "What color car did the suspect drive?"
)

print(missing_answer)

The available case files do not contain that information.
